# COMP0005 - GROUP COURSEWORK
# Experimental Evaluation of Search Data Structures and Algorithms

The cell below defines **AbstractSearchInterface**, an interface to support basic insert/search operations; you will need to implement this three times, to realise your three search data structures of choice among: (1) *2-3 Tree*, (2) *AVL Tree*, (3) *LLRB BST*; (4) *B-Tree*; and (5) *Scapegoat Tree*. <br><br>**Do NOT modify the next cell** - use the dedicated cells further below for your implementation instead. <br>

In [ ]:
# DO NOT MODIFY THIS CELL

from abc import ABC, abstractmethod  

class AbstractSearchInterface(ABC):
    '''
    Abstract class to support search/insert operations (plus underlying data structure)
    
    '''
        
    @abstractmethod
    def insertElement(self, element):     
        '''
        Insert an element in a search tree
            Parameters:
                    element: string to be inserted in the search tree (string)

            Returns:
                    "True" after successful insertion, "False" if element is already present (bool)
        '''
        
        pass 
    

    @abstractmethod
    def searchElement(self, element):
        '''
        Search for an element in a search tree
            Parameters:
                    element: string to be searched in the search tree (string)

            Returns:
                    "True" if element is found, "False" otherwise (bool)
        '''

        pass

Use the cell below to define any auxiliary data structure and python function you may need. Leave the implementation of the main API to the next code cells instead.

In [ ]:
class TwoThreeNode:
    def __init__(self, val1, val2=None):
        self.left_data, self.right_data = val1, val2
        self.left, self.middle, self.right = None, None, None

class LLRBNode:
    def __init__(self, key, color):
        self.key = key
        self.color = color
        self.left = None
        self.right = None

class ScapegoatNode:
    def __init__(self, value, parent=None):
        self.value: str = value
        self.left: ScapegoatNode | None = None
        self.right: ScapegoatNode | None = None
        self.parent: ScapegoatNode | None = parent
        self.size = 1

Use the cell below to implement the requested API by means of **2-3 Tree** (if among your chosen data structure).

In [ ]:
class TwoThreeTree(AbstractSearchInterface):
    def __init__(self):
        self.head = None

    def insertElement(self, element):
        # Returns: (promoted_key, left_child, right_child, inserted)
        def insert_recursive(node, key):
            if node.left_data == key or node.right_data == key:
                return None, None, None, False

            if node.left is None and node.middle is None and node.right is None:
                # Leaf with one key
                if node.right_data is None:
                    if key < node.left_data:
                        node.left_data, node.right_data = key, node.left_data
                    else:
                        node.left_data, node.right_data = node.left_data, key
                    return None, None, None, True

                # Leaf with two keys - split and promote middle
                smallest, middle, largest = sorted((node.left_data, node.right_data, key))
                node.left_data = smallest
                node.right_data = None
                new_right_node = TwoThreeNode(largest)
                return middle, node, new_right_node, True

            if key < node.left_data:
                branch = "left"
                child = node.left
            elif node.right_data is None or key < node.right_data:
                branch = "middle"
                child = node.middle
            else:
                branch = "right"
                child = node.right

            promoted, left_child, right_child, inserted = insert_recursive(child, key)
            if not inserted or promoted is None:
                return None, None, None, inserted

            # Parent with one key: absorb the promoted key
            if node.right_data is None:
                if branch == "left":
                    original_left_key = node.left_data
                    original_middle = node.middle
                    node.left_data = promoted
                    node.right_data = original_left_key
                    node.left = left_child
                    node.middle = right_child
                    node.right = original_middle
                else:
                    node.right_data = promoted
                    node.middle = left_child
                    node.right = right_child
                return None, None, None, True

            # Parent with two keys: split the parent
            old_left = node.left
            old_middle = node.middle
            old_right = node.right
            old_key1 = node.left_data
            old_key2 = node.right_data

            if branch == "left":
                promoted_upward = old_key1
                node.left_data = promoted
                node.right_data = None
                node.left = left_child
                node.middle = right_child
                node.right = None
                new_right_parent = TwoThreeNode(old_key2)
                new_right_parent.left = old_middle
                new_right_parent.middle = old_right
            elif branch == "middle":
                promoted_upward = promoted
                node.left_data = old_key1
                node.right_data = None
                node.left = old_left
                node.middle = left_child
                node.right = None
                new_right_parent = TwoThreeNode(old_key2)
                new_right_parent.left = right_child
                new_right_parent.middle = old_right
            else:
                promoted_upward = old_key2
                node.left_data = old_key1
                node.right_data = None
                node.left = old_left
                node.middle = old_middle
                node.right = None
                new_right_parent = TwoThreeNode(promoted)
                new_right_parent.left = left_child
                new_right_parent.middle = right_child

            return promoted_upward, node, new_right_parent, True

        if self.head is None:
            self.head = TwoThreeNode(element)
            return True

        promoted, left_subtree, right_subtree, inserted = insert_recursive(self.head, element)
        if inserted and promoted is not None:
            self.head = TwoThreeNode(promoted)
            self.head.left = left_subtree
            self.head.middle = right_subtree
        return inserted

    def searchElement(self, element):
        curr = self.head
        while curr:
            if curr.left_data == element or curr.right_data == element:
                return True
            if element < curr.left_data:
                curr = curr.left
            elif curr.right_data is None or element < curr.right_data:
                curr = curr.middle
            else:
                curr = curr.right
        return False

Use the cell below to implement the requested API by means of **LLRB BST** (if among your chosen data structure).

In [ ]:
class LLRBBST(AbstractSearchInterface):
    RED = True
    BLACK = False

    def __init__(self):
        self.root = None

    def _is_red(self, node):
        return node is not None and node.color == self.RED

    def _rotate_left(self, h):
        x = h.right
        h.right = x.left
        x.left = h
        x.color = h.color
        h.color = self.RED
        return x

    def _rotate_right(self, h):
        x = h.left
        h.left = x.right
        x.right = h
        x.color = h.color
        h.color = self.RED
        return x

    def _flip_colors(self, h):
        h.color = self.RED
        if h.left is not None:
            h.left.color = self.BLACK
        if h.right is not None:
            h.right.color = self.BLACK
        
    def insertElement(self, element):
        inserted = False
        def _insert(h, key):
            nonlocal inserted
            if h is None:
                inserted = True
                return LLRBNode(key, self.RED)

            if key < h.key:
                h.left = _insert(h.left, key)
            elif key > h.key:
                h.right = _insert(h.right, key)
            else:
                return h

            if self._is_red(h.right) and not self._is_red(h.left):
                h = self._rotate_left(h)
            if self._is_red(h.left) and self._is_red(h.left.left):
                h = self._rotate_right(h)
            if self._is_red(h.left) and self._is_red(h.right):
                self._flip_colors(h)

            return h

        self.root = _insert(self.root, element)
        if self.root is not None:
            self.root.color = self.BLACK
 
        return inserted
    
    

    def searchElement(self, element):     
        found = False
        current = self.root

        while current is not None:
            if element < current.key:
                current = current.left
            elif element > current.key:
                current = current.right
            else:
                found = True
                break
        
        return found  

Use the cell below to implement the requested API by means of **Scapegoat Tree** (if among your chosen data structure).

In [ ]:
class ScapegoatTree(AbstractSearchInterface):

    def __init__(self):
        self.root = None
        self.tree_size = 0
        self._ln_inv_alpha = 0.35667494393 # Precomputed ln(1/alpha) for alpha=0.7
        self.a_weight = 0.7
    def _max_depth(self, n):
        if n <= 1:
            return 0.0
        # log_{1/alpha}(n) = ln(n) / ln(1/alpha)
        # Use bit_length for ln(n): ln(n) = log2(n) * ln(2)
        LN2 = 0.69314718056
        shift = n.bit_length() - 1 # equivalent to floor(log2(n))
        m = n / (1 << shift) # n / 2^shift, which is in [1, 2)
        y = (m - 1.0) / (m + 1.0)
        y2 = y * y
        # ln(n) = ln(2) * shift * ln(m) where m is in [1, 2)
        # ln(m) calculated using the area hyperbolic tangent series expansion
        ln_n = LN2 * (shift + 2.0 * y * (1.0 + y2 * (1.0/3 + y2 * (1.0/5 + y2 * 1.0/7))))
        return ln_n / self._ln_inv_alpha
    
    def get_size(self, node):
        if node is None:
            return 0
        return 1 + self.get_size(node.left) + self.get_size(node.right)

    def find_scapegoat(self, element):
        child = element
        child_size = 1
        parent = child.parent

        while parent is not None:
            parent_size = parent.size  # Use the stored size attribute
            if child_size > self.a_weight * parent_size:
                return parent
            child = parent
            child_size = parent_size
            parent = parent.parent

        return None

    def collect_elements(self, node, result=None):
        if result is None:
            result = []
        if node is None:
            return result
        self.collect_elements(node.left, result)
        result.append(node.value)
        self.collect_elements(node.right, result)
        return result

    def rebuild_subtree(self, elements, lo, hi, parent=None):
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        node = ScapegoatNode(elements[mid], parent=parent)
        node.left = self.rebuild_subtree(elements, lo, mid - 1, parent=node)
        node.right = self.rebuild_subtree(elements, mid + 1, hi, parent=node)
        node.size = 1 + (node.left.size if node.left else 0) + (node.right.size if node.right else 0)
        return node

    def insertElement(self, element):
        inserted_node = None
        depth = 0
        current = self.root
        if self.root is None:
            self.root = ScapegoatNode(element)
            self.tree_size = 1
            return True
        while current is not None:
            current.size += 1  # Increment size for all ancestors
            if element == current.value:
                current.size -= 1 # Revert size increment for duplicates
                return False  # Duplicate, do not insert
            elif element < current.value:
                if current.left is None:
                    current.left = ScapegoatNode(element, parent=current)
                    inserted_node = current.left
                    depth += 1
                    break
                else:
                    current = current.left
                    depth += 1
            else:
                if current.right is None:
                    current.right = ScapegoatNode(element, parent=current)
                    inserted_node = current.right
                    depth += 1
                    break
                else:
                    current = current.right
                    depth += 1
        self.tree_size += 1
        if inserted_node is not None and depth > self._max_depth(self.tree_size):
            scapegoat = self.find_scapegoat(inserted_node)
            # Rebuild the subtree rooted at the scapegoat node to be perfectly balanced
            if scapegoat:
                elements = self.collect_elements(scapegoat)
                rebuilt = self.rebuild_subtree(elements, 0, len(elements) - 1, parent=scapegoat.parent)

                if scapegoat.parent is None:
                    self.root = rebuilt
                elif scapegoat.parent.left == scapegoat:
                    scapegoat.parent.left = rebuilt
                else:
                    scapegoat.parent.right = rebuilt
        return True

    def searchElement(self, element):
        current = self.root
        while current is not None:
            if element == current.value:
                return True
            elif element < current.value:
                current = current.left
            else:
                current = current.right
        return False

Use the cell below to implement the **synthetic data generator** needed by your experimental framework (be mindful of code readability and reusability).

In [ ]:
import string
import random

class TestDataGenerator():
    '''
    A class to represent a synthetic data generator.

    ...

    Attributes
    ----------
    seed : int
        Random seed for reproducibility across experiments.

    Methods
    -------
    random_strings(n, length=8)
        Generate n random strings of given length (uniform distribution).
    sorted_strings(n, length=8)
        Generate n random strings in ascending sorted order.
    reverse_sorted_strings(n, length=8)
        Generate n random strings in descending sorted order.
    skewed_strings(n, length=8, prefix_chars=3)
        Generate n strings sharing a common prefix (simulates clustering).
    generate(n, distribution='random', length=8)
        Main entry point — returns a dataset by distribution type.
    '''

    def __init__(self, seed=42):
        self.seed = seed
        random.seed(seed)

    def _random_string(self, length):
        '''Generate a single random lowercase alphabetic string.'''
        return ''.join(random.choices(string.ascii_lowercase, k=length))

    def random_strings(self, n, length=8):
        '''Generate n uniformly random strings — typical average-case input.'''
        return [self._random_string(length) for _ in range(n)]

    def sorted_strings(self, n, length=8):
        '''Generate n random strings sorted in ascending order — tests sorted insertion.'''
        return sorted(self.random_strings(n, length))

    def reverse_sorted_strings(self, n, length=8):
        '''Generate n random strings sorted in descending order — adversarial input.'''
        return sorted(self.random_strings(n, length), reverse=True)

    def skewed_strings(self, n, length=8, prefix_chars=3):
        '''
        Generate n strings that share a short common prefix.
        Simulates real-world key clustering (e.g. usernames, file paths).
        '''
        prefix = ''.join(random.choices(string.ascii_lowercase, k=prefix_chars))
        suffix_length = length - prefix_chars
        return [prefix + self._random_string(suffix_length) for _ in range(n)]

    def generate(self, n, distribution='random', length=8):
        '''
        Main entry point for dataset generation.

        Parameters
        ----------
        n : int
            Number of strings to generate.
        distribution : str
            One of 'random', 'sorted', 'reverse', 'skewed'.
        length : int
            Length of each generated string.

        Returns
        -------
        list of str
            A list of n unique strings (duplicates removed).
        '''
        if distribution == 'random':
            data = self.random_strings(n, length)
        elif distribution == 'sorted':
            data = self.sorted_strings(n, length)
        elif distribution == 'reverse':
            data = self.reverse_sorted_strings(n, length)
        elif distribution == 'skewed':
            data = self.skewed_strings(n, length)
        else:
            raise ValueError(f"Unknown distribution '{distribution}'. "
                             f"Choose from: 'random', 'sorted', 'reverse', 'skewed'.")

        # Remove duplicates while preserving order
        seen = set()
        unique = []
        for s in data:
            if s not in seen:
                seen.add(s)
                unique.append(s)
        return unique

Use the cell below to implement the requested **experimental framework** (be mindful of code readability and reusability).

In [ ]:
import timeit
from matplotlib import pyplot as plt

class ExperimentalFramework():
    '''
    A class to represent an experimental framework.

    ...

    Attributes
    ----------
        data_generator (TestDataGenerator)
            An instance of the TestDataGenerator class to create datasets for testing.
        search_structures (list)
            A list of search structures to be tested.

    Methods
    -------
        test_insertion(structure, dataset)
            Test the time taken to insert all elements of a dataset into a search structure.
        test_search(structure, search_samples)
            Test the time taken to search for a list of sample elements in a search structure.
        run_experiments(n_values, distributions, test_count)
            Run experiments for each search structure, dataset size, and distribution type.
        plot_results(results)
            Plot the timing results using matplotlib.

    '''

    def __init__(self, data_generator, search_structures):
        self.data_generator = data_generator
        self.search_structures = search_structures
    
    def test_insertion(self, structure, dataset):
        '''Test the time taken to insert all elements of a dataset into a search structure.'''
        start_time = timeit.default_timer()
        for item in dataset:
            structure.insertElement(item)
        end_time = timeit.default_timer()
        return end_time - start_time
    
    def test_search(self, structure, search_samples):
        '''Test the time taken to search for a list of sample elements in a search structure.'''
        start_time = timeit.default_timer()
        for item in search_samples:
            structure.searchElement(item)
        end_time = timeit.default_timer()
        return end_time - start_time
    
    def run_experiments(self, n_values, distributions, test_count):
        '''
        Run experiments for each search structure, dataset size, and distribution type.

        Parameters
        ----------
        n_values : list[int]
            List of dataset sizes to test (e.g. [1000, 5000, 10000]).
        distributions : list[str]
            List of distribution types to test (e.g. ['random', 'sorted', 'reverse', 'skewed']).

        Returns
        -------
        dict
            A nested dictionary containing timing results structured as:
                results[structure_name][distribution][n] = time_taken
        '''
        results_list = []
        for i in range(test_count):
            print(f"Running trial {i+1}/{test_count}...")
            results = {}
            for structure in self.search_structures:
                structure_name = type(structure).__name__
                results[structure_name] = {}
                for dist in distributions:
                    dist_results = results[structure_name][dist] = {}
                    for n in n_values:
                        dataset = self.data_generator.generate(n, distribution=dist)
                        structure = type(structure)()
                        
                        # -- INSERTION --
                        time_taken = self.test_insertion(structure, dataset)
                        print(f"Inserted {n} items into {structure_name} with {dist} distribution in {time_taken:.4f} seconds.")
                        dist_results[f"{n}_insert"] = time_taken

                        # -- SEARCHING EXISTING ELEMENTS --
                        search_samples = random.sample(dataset, int(0.5 * len(dataset)))  # Sample 50% of elements to search for
                        time_taken = self.test_search(structure, search_samples)
                        dist_results[f"{n}_search_success"] = time_taken
                        print(f"Searched {len(search_samples)} items in {structure_name} with {dist} distribution in {time_taken:.4f} seconds.")

                        # -- SEARCHING NON-EXISTENT ELEMENTS --
                        non_existent_samples = [self.data_generator._random_string(8) for _ in range(len(search_samples))] # Set number of non-existent samples equal to number of successful search samples for balanced testing
                        time_taken = self.test_search(structure, non_existent_samples)
                        dist_results[f"{n}_search_failure"] = time_taken
                        print(f"Searched {len(non_existent_samples)} non-existent items in {structure_name} with {dist} distribution in {time_taken:.4f} seconds.\n") 
            results_list.append(results)
        return results_list

    def plot_results(self, all_results):
        '''
        Plot the timing results using matplotlib.
        
        Parameters
        ----------
        all_results : list[dict]
            A list of result dictionaries from multiple trials, structured as:
                results[structure_name][distribution][n] = time_taken
        '''
        distributions = list(next(iter(all_results[0].values())).keys())
        structure_names = list(all_results[0].keys())

        metrics = {
            'insert':         ('Insertion Time',         'Time Taken (seconds)'),
            'search_success': ('Successful Search Time', 'Time Taken (seconds)'),
            'search_failure': ('Failed Search Time',     'Time Taken (seconds)'),
        }

        def median(vals):
            s = sorted(vals)
            n = len(s)
            mid = n // 2
            return (s[mid - 1] + s[mid]) / 2 if n % 2 == 0 else s[mid]

        def percentile(vals, p):
            s = sorted(vals)
            idx = (len(s) - 1) * p / 100
            lo, hi = int(idx), min(int(idx) + 1, len(s) - 1)
            return s[lo] + (s[hi] - s[lo]) * (idx - lo)

        for dist in distributions:
            for metric_key, (metric_label, y_label) in metrics.items():
                fig, ax = plt.subplots(figsize=(10, 6))

                for structure_name in structure_names:
                    n_values = sorted([
                        int(k.split('_')[0])
                        for k in all_results[0][structure_name][dist]
                        if k.endswith(f'_{metric_key}')
                    ])

                    medians, p25, p75 = [], [], []
                    for n in n_values:
                        key = f"{n}_{metric_key}"
                        trial_vals = [
                            trial[structure_name][dist][key]
                            for trial in all_results
                            if key in trial[structure_name][dist]
                        ]
                        medians.append(median(trial_vals))
                        p25.append(percentile(trial_vals, 25))
                        p75.append(percentile(trial_vals, 75))

                    line, = ax.plot(n_values, medians, marker='o', label=structure_name)
                    ax.fill_between(n_values, p25, p75, alpha=0.2, color=line.get_color())

                ax.set_title(f'{metric_label} vs Dataset Size ({dist} distribution)')
                ax.set_xlabel('Dataset Size (n)')
                ax.set_ylabel(y_label)
                ax.set_xscale('log')
                ax.set_yscale('log')
                ax.legend()
                ax.grid(True, which='both', linestyle='--', alpha=0.5)
                plt.tight_layout()
                plt.show()
            

    

Use the cell below to illustrate the python code you used to **fully evaluate** your three chosen search data structures and algortihms. The code below should illustrate, for example, how you made used of the **TestDataGenerator** class to generate test data of various size and properties; how you instatiated the **ExperimentalFramework** class to  evaluate each data structure using such data, collect information about their execution time, plot results, etc. Any results you illustrate in the companion PDF report should have been generated using the code below.

In [ ]:
framework = ExperimentalFramework(
    data_generator=TestDataGenerator(seed=42),
    search_structures=[TwoThreeTree(), LLRBBST(), ScapegoatTree()]
)

results = framework.run_experiments(
    n_values=[100, 500, 1000, 5000, 10000, 50000],
    distributions=['random', 'sorted', 'reverse', 'skewed'],
    test_count=25
)

framework.plot_results(results)